# 🧪 Notebook de Prueba: Enriquecimiento IA de Actividades

Este notebook permite probar paso a paso el enriquecimiento de actividades con campos de IA:
- `es_aire_libre`: Clasificación booleana
- `edad_minima`: Extracción de edad mínima
- `edad_maxima`: Extracción de edad máxima

## Requisitos previos
0. Entorno 
# Código: 
```
# Verificar versión
python --version

# Crear entorno virtual con Python 3.11
python3.11 -m venv venv_ia

# Activar
# source venv_ia/bin/activate  # Linux/Mac
venv_ia\Scripts\activate     # Windows

# Instalar dependencias
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install -r requirements-ia.txt
```

1. Instalar dependencias: `pip install -r requirements-ia.txt`
2. Tener el archivo `actividades_procesadas.json` en la carpeta `data/`

## Paso 1: Configuración e Imports

In [5]:
import sys
import os
import json
import time

# Añadir scripts al path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

from models.zero_shot import AirClassifier
from models.qa_age import AgeExtractor
from utils.text_processor import clean_text

print("✅ Imports cargados correctamente")

✅ Imports cargados correctamente


## Paso 2: Cargar actividades de prueba

In [6]:
# Ruta al archivo de actividades
DATA_PATH = os.path.join('..', 'data', 'actividades_procesadas.json')

# Cargar actividades
if os.path.exists(DATA_PATH):
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        actividades = json.load(f)
    print(f"✅ Cargadas {len(actividades)} actividades")
    
    # Mostrar primera actividad como ejemplo
    if actividades:
        print("\n📋 Primera actividad:")
        print(f"   Título: {actividades[0].get('title', 'N/A')}")
        print(f"   Categoría: {actividades[0].get('category', 'N/A')}")
        print(f"   Descripción: {actividades[0].get('description', 'N/A')[:100]}...")
else:
    print(f"❌ No se encontró el archivo: {DATA_PATH}")
    print("   Asegúrate de tener el archivo en la carpeta data/")

✅ Cargadas 1007 actividades

📋 Primera actividad:
   Título: 10º Festival Internacional de Folclore de Latina.Folkfestival 'Ara de Madrid'
   Categoría: Fiestas
   Descripción: 10º Festival Internacional de Folclore de Latina – Folkfestival 'Ara de Madrid', con la participació...


## Paso 3: Seleccionar actividades de prueba

Seleccionamos un subconjunto pequeño para probar rápidamente

In [6]:
# Número de actividades a probar
N_PRUEBA = 5

# Seleccionar primeras N actividades
actividades_prueba = actividades[:N_PRUEBA]

print(f"🧪 Actividades seleccionadas para prueba: {len(actividades_prueba)}")
for i, act in enumerate(actividades_prueba, 1):
    print(f"\n{i}. {act.get('title', 'Sin título')}")
    print(f"   Desc: {act.get('description', 'Sin descripción')[:80]}...")

🧪 Actividades seleccionadas para prueba: 5

1. 10º Festival Internacional de Folclore de Latina.Folkfestival 'Ara de Madrid'
   Desc: 10º Festival Internacional de Folclore de Latina – Folkfestival 'Ara de Madrid',...

2. 3deVermu Funk &amp; Disco DJset a vinilo
   Desc: '3deVermu Funk &amp;amp; Disco DJset a vinilo' (Asoc. Scout Jamboree)...

3. 4 escenas, 4 estilos
   Desc: ...

4. 6 x 4 y Cueros
   Desc: Música. FIGUE 2026    Violín: José Gabriel Nunes  Percusión: Devis Colmenares  G...

5. 75 años de dirección artística cinematográfica. Gil Parrondo, creador de sueños
   Desc: La muestra sintetiza el proceso creativo del primer español ganador de un Oscar ...


## Paso 4: Inicializar modelos de IA

In [7]:
print("🤖 Inicializando modelos...")
print("   Esto puede tardar 1-2 minutos la primera vez (descarga de modelos)")

# Inicializar clasificador de aire libre
air_classifier = AirClassifier(device="cpu")

# Inicializar extractor de edades
age_extractor = AgeExtractor(device="cpu")

print("\n✅ Modelos cargados y listos")

🤖 Inicializando modelos...
   Esto puede tardar 1-2 minutos la primera vez (descarga de modelos)
🤖 Cargando modelo de clasificación: distilbert-base-uncased-finetuned-sst-2-english


d:\environments\venv_ia\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alberto\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\environments\venv_ia\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecate

✅ Modelo cargado en cpu
🤖 Cargando modelo QA: distilbert-base-cased-distilled-squad


✅ Modelo QA cargado en cpu

✅ Modelos cargados y listos


## Paso 5: Probar clasificación "es_aire_libre"

In [12]:
print("🌳 Probando clasificación de 'aire libre'...\n")

for act in actividades_prueba:
    title = act.get('title', '')
    description = act.get('description', '')
    
    # Limpiar texto
    clean_desc = clean_text(description)
    
    # Clasificar
    start = time.time()
    es_aire_libre = air_classifier.classify_single(clean_desc, title)
    elapsed = time.time() - start
    
    print(f"📍 {title[:50]}...")
    print(f"    Descripción: {clean_desc}...")
    print(f"   🌳 Aire libre: {es_aire_libre} ({elapsed:.2f}s)")
    print(f"   📝 Desc: {clean_desc[:60]}...\n")

🌳 Probando clasificación de 'aire libre'...

📍 10º Festival Internacional de Folclore de Latina.F...
    Descripción: 10º Festival Internacional de Folclore de Latina  Folkfestival Ara de Madrid, con la participación de los grupos: Grupo Folclórico Villaviciosa-Aires de Asturias (Asturias) bajo la dirección de Elena Alonso. Grupo Leyendas de México (México), bajo la dirección de Ana Paola de la Cruz. Ballet Ara de Madrid, bajo la dirección de Carmina Villar....
   🌳 Aire libre: True (0.21s)
   📝 Desc: 10º Festival Internacional de Folclore de Latina  Folkfestiv...

📍 3deVermu Funk &amp; Disco DJset a vinilo...
    Descripción: 3deVermu Funk amp;amp; Disco DJset a vinilo (Asoc. Scout Jamboree)...
   🌳 Aire libre: False (0.09s)
   📝 Desc: 3deVermu Funk amp;amp; Disco DJset a vinilo (Asoc. Scout Jam...

📍 4 escenas, 4 estilos...
    Descripción: ...
   🌳 Aire libre: False (0.05s)
   📝 Desc: ...

📍 6 x 4 y Cueros...
    Descripción: Música. FIGUE 2026 Violín: José Gabriel Nunes Percusión: 

## Paso 6: Probar extracción de edades

In [10]:
print("👶 Probando extracción de edades...\n")

for act in actividades_prueba:
    title = act.get('title', '')
    description = act.get('description', '')
    
    # Limpiar texto
    clean_desc = clean_text(description)
    
    # Extraer edades
    start = time.time()
    edades = age_extractor.extract_ages(clean_desc, title)
    elapsed = time.time() - start
    
    print(f"📍 {title[:50]}...")
    print(f"   Descripción: {clean_desc}")
    print(f"   👶 Edad mínima: {edades['edad_minima']}")
    print(f"   👴 Edad máxima: {edades['edad_maxima']}")
    print(f"   ⏱️ Tiempo: {elapsed:.2f}s\n")

👶 Probando extracción de edades...

📍 10º Festival Internacional de Folclore de Latina.F...
   Descripción: 10º Festival Internacional de Folclore de Latina  Folkfestival Ara de Madrid, con la participación de los grupos: Grupo Folclórico Villaviciosa-Aires de Asturias (Asturias) bajo la dirección de Elena Alonso. Grupo Leyendas de México (México), bajo la dirección de Ana Paola de la Cruz. Ballet Ara de Madrid, bajo la dirección de Carmina Villar.
   👶 Edad mínima: 10
   👴 Edad máxima: None
   ⏱️ Tiempo: 1.20s

📍 3deVermu Funk &amp; Disco DJset a vinilo...
   Descripción: 3deVermu Funk amp;amp; Disco DJset a vinilo (Asoc. Scout Jamboree)
   👶 Edad mínima: None
   👴 Edad máxima: None
   ⏱️ Tiempo: 0.66s

📍 4 escenas, 4 estilos...
   Descripción: 
   👶 Edad mínima: 4
   👴 Edad máxima: None
   ⏱️ Tiempo: 0.23s

📍 6 x 4 y Cueros...
   Descripción: Música. FIGUE 2026 Violín: José Gabriel Nunes Percusión: Devis Colmenares Guitarra: Jonnathan El Barouki Edad recomendada: público adulto Repar

In [41]:
import os
import json
import time
from transformers import pipeline

# Nota: Asumo que la función load_json está definida en otra parte de tu script 
# o importada. La dejo comentada como referencia si hiciera falta:
# from utils import load_json 

# def procesar_actividades(archivo_origen, archivo_destino="./data/resultados_clasificados.json"):

# 1. Inicializar los pipelines locales de Hugging Face
# La primera vez se descargarán automáticamente al disco (~1.5 GB total)
print("Cargando modelos Transformers en memoria local...")

# clasificador_local = pipeline(
#     "zero-shot-classification", 
#     model="facebook/bart-large-mnli"
# )

extractor_local = pipeline(
    "question-answering", 
    # model="mrm8488/bert-multi-cased-finetuned-xquadv1"
    model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es"
)

# # 2. Leer tu archivo JSON original utilizando tu función load_json
# actividades = load_json(archivo_origen, [])
# if not isinstance(actividades, list):
#     print(f"Formato inesperado en {archivo_origen}: se esperaba una lista.")
#     return

actividades = actividades_prueba
# Tu bucle exacto de extracción de textos
lista_textos = []
for activity in actividades:
    texto = activity.get('description', '').strip()
    lista_textos.append(texto)
    
print(f"Iniciando el procesamiento local de {len(lista_textos)} textos...")
tiempo_inicio = time.time()

# Definimos las categorías semánticas objetivo
etiquetas = ["actividad al aire libre", "resto"]

# 3. Procesamiento local registro a registro
for idx, texto in enumerate(lista_textos):
    if not texto:
        actividades[idx]['categoria'] = "N/A"
        actividades[idx]['edad_minima'] = "N/A"
        actividades[idx]['edad_maxima'] = "N/A"
        continue

    # --- PASO A: CLASIFICACIÓN ZERO-SHOT LOCAL ---
    try:
        res_clase = clasificador_local(texto, candidate_labels=etiquetas)
        # El pipeline local devuelve un diccionario. Extraemos la primera etiqueta (la de mayor score)
        categoria_asignada = res_clase['labels'][0]
    except Exception as e:
        categoria_asignada = f"Error Local: {str(e)}"

    # --- PASO B: EXTRACCIÓN DE EDAD MÍNIMA LOCAL ---
    try:
        res_min = extractor_local(
            question="¿Cual es la edad recomendada del evento? Buscamos si hay edad mínima. Adulto se considera a partir de 18 años. Niños se consideran menores de 12 años.",
            context=texto
        )
        # Filtramos por confianza (score) de la misma forma que hacías con el cliente de la API
        edad_minima = res_min['answer'] if res_min and res_min['score'] > 0.1 else "No mencionada"
    except Exception:
        edad_minima = "No mencionada"

    # --- PASO C: EXTRACCIÓN DE EDAD MÁXIMA LOCAL ---
    try:
        res_max = extractor_local(
            question="¿Cual es la edad recomendada del evento? Buscamos si hay edad máxima. Adulto se considera a partir de 18 años. Niños se consideran menores de 12 años.",
            context=texto
        )
        edad_maxima = res_max['answer'] if res_max and res_max['score'] > 0.1 else "No mencionada"
    except Exception:
        edad_maxima = "No mencionada"

    # Inyectamos los resultados procesados directamente en el objeto original de tu JSON
    actividades[idx]['categoria'] = categoria_asignada
    actividades[idx]['edad_minima'] = edad_minima
    actividades[idx]['edad_maxima'] = edad_maxima

    # NOTA: Se elimina el time.sleep(1.0) y los bloques de reintentos por error HTTP (HfHubHTTPError)
    # dado que en local no hay Rate Limits ni tiempos de espera de carga de la API.

    if (idx + 1) % 50 == 0:
        print(f"Progreso: {idx + 1}/{len(lista_textos)} registros procesados locales.")

    print(f"📍 ID: {actividades[idx]['id']}...")
    print(f"📍 Título: {actividades[idx]['title']}...")
    print(f"   Descripción: {actividades[idx]['description']}...")
    print(f"   Categoría: {actividades[idx]['categoria']}")
    print(f"   👶 Edad mínima: {actividades[idx]['edad_minima']}")
    print(f"   👴 Edad máxima: {actividades[idx]['edad_maxima']}")
    print(f"   ⏱️ Tiempo: {elapsed:.2f}s\n")
# # Asegurar que el directorio de destino exista antes de guardar
# os.makedirs(os.path.dirname(archivo_destino), exist_ok=True)

# # 4. Guardar el JSON resultante manteniendo todo tu contenido intacto
# with open(archivo_destino, "w", encoding="utf-8") as f:
#     json.dump(actividades, f, ensure_ascii=False, indent=2)
    
# tiempo_total = time.time() - tiempo_inicio
# print(f"¡Proceso completado en {tiempo_total:.2f} segundos! Archivo estructurado guardado en '{archivo_destino}'.")


Cargando modelos Transformers en memoria local...


Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Iniciando el procesamiento local de 5 textos...
📍 ID: https://datos.madrid.es/egob/catalogo/tipo/evento/12008124-10-festival-internacional-folclore-latina-folkfestival-ara-madrid-.json...
📍 Título: 10º Festival Internacional de Folclore de Latina.Folkfestival 'Ara de Madrid'...
   Descripción: 10º Festival Internacional de Folclore de Latina – Folkfestival 'Ara de Madrid', con la participación de los grupos:  Grupo Folclórico 'Villaviciosa-Aires de Asturias' (Asturias) bajo la dirección de Elena Alonso.  Grupo 'Leyendas de México' (México), bajo la dirección de Ana Paola de la Cruz.  Ballet 'Ara de Madrid', bajo la dirección de Carmina Villar....
   Categoría: actividad al aire libre
   👶 Edad mínima: No mencionada
   👴 Edad máxima: No mencionada
   ⏱️ Tiempo: 0.19s

📍 ID: https://datos.madrid.es/egob/catalogo/tipo/evento/12366331-3devermu-funk-disco-djset-vinilo.json...
📍 Título: 3deVermu Funk &amp; Disco DJset a vinilo...
   Descripción: '3deVermu Funk &amp;amp; Disco DJset a vinilo'

KeyboardInterrupt: 

In [ ]:
def process_activity_id(id):
    
    # Clasifica una actividad concreta para ver el resultado de forma detallada:
    for activity in actividades:
        # print(f"Revisando ID: {activity['id']}...")
        if activity['app_id'] == id:
            print(f"\n🔍 Procesando actividad con ID: {activity['app_id']} - {activity['title']}\n")
            # Obtenemos el texto de la descripción para procesarlo con los pipelines locales:
            texto = activity.get('description', '').strip()
            
            try:
                res_clase = clasificador_local(texto, candidate_labels=etiquetas)
                # El pipeline local devuelve un diccionario. Extraemos la primera etiqueta (la de mayor score)
                categoria_asignada = res_clase['labels'][0]
            except Exception as e:
                categoria_asignada = f"Error Local: {str(e)}"

            # --- PASO B: EXTRACCIÓN DE EDAD MÍNIMA LOCAL ---
            try:
                res_min = extractor_local(
                    question="edad mínima / a partir de / recomendado desde / mayores de",
                    context=texto
                )
                # Filtramos por confianza (score) de la misma forma que hacías con el cliente de la API
                edad_minima = res_min['answer'] if res_min and res_min['score'] > 0.1 else "No mencionada"
            except Exception:
                edad_minima = "No mencionada"

            # --- PASO C: EXTRACCIÓN DE EDAD MÁXIMA LOCAL ---
            try:
                res_max = extractor_local(
                    question="edad máxima / hasta / recomendado hasta / menores de",
                    context=texto
                )
                edad_maxima = res_max['answer'] if res_max and res_max['score'] > 0.1 else "No mencionada"
            except Exception:
                edad_maxima = "No mencionada"

            # Inyectamos los resultados procesados directamente en el objeto original de tu JSON
            activity['categoria'] = categoria_asignada
            activity['edad_minima'] = edad_minima
            activity['edad_maxima'] = edad_maxima

            # NOTA: Se elimina el time.sleep(1.0) y los bloques de reintentos por error HTTP (HfHubHTTPError)
            # dado que en local no hay Rate Limits ni tiempos de espera de carga de la API.

            if (idx + 1) % 50 == 0:
                print(f"Progreso: {idx + 1}/{len(lista_textos)} registros procesados locales.")

            print(f"📍 ID: {activity['id']}...")
            print(f"📍 Título: {activity['title']}...")
            print(f"   Descripción: {activity['description']}...")
            print(f"   Categoría: {activity['categoria']}")
            print(f"   👶 Edad mínima: {activity['edad_minima']}")
            print(f"   👴 Edad máxima: {activity['edad_maxima']}")
            print(f"   ⏱️ Tiempo: {elapsed:.2f}s\n")

In [7]:
# Ruta al archivo de actividades
DATA_PATH = os.path.join('..', 'data', 'actividades_procesadas.json')

# Cargar actividades
if os.path.exists(DATA_PATH):
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        actividades = json.load(f)
    print(f"✅ Cargadas {len(actividades)} actividades")
    
    # Mostrar primera actividad como ejemplo
    if actividades:
        print("\n📋 Primera actividad:")
        print(f"   Título: {actividades[0].get('title', 'N/A')}")
        print(f"   Categoría: {actividades[0].get('category', 'N/A')}")
        print(f"   Descripción: {actividades[0].get('description', 'N/A')[:100]}...")
else:
    print(f"❌ No se encontró el archivo: {DATA_PATH}")
    print("   Asegúrate de tener el archivo en la carpeta data/")

✅ Cargadas 1007 actividades

📋 Primera actividad:
   Título: 10º Festival Internacional de Folclore de Latina.Folkfestival 'Ara de Madrid'
   Categoría: Fiestas
   Descripción: 10º Festival Internacional de Folclore de Latina – Folkfestival 'Ara de Madrid', con la participació...


In [1]:
import torch
from transformers import pipeline

# Inicialización global del LLM local ligero
llm_local = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

d:\environments\venv_ia\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
d:\environments\venv_ia\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alberto\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pytho

In [ ]:
def process_activity_id(id):
    # Clasifica una actividad concreta para ver el resultado de forma detallada:
    for activity in actividades:
        if activity['app_id'] == id:
            start_time = time.time()
            print(f"\n🔍 Procesando actividad con ID: {activity['app_id']} - {activity['title']}\n")
            
            texto = activity.get('description', '').strip()
            
            if not texto:
                activity['categoria'] = "N/A"
                activity['edad_minima'] = "No mencionada"
                activity['edad_maxima'] = "No mencionada"
            else:
                # Estructuramos el prompt del sistema para el LLM local
                # prompt_sistema = ("Eres un analizador de texto estricto. Tu tarea es extraer información en español. "
                #             "Devuelve ÚNICAMENTE un objeto JSON plano con tres claves exactas: "
                #             "'categoria' (debe ser obligatoriamente 'actividad al aire libre' o 'resto'), "
                #             "'edad_minima' (debe ser un número entero. Si el texto indica 'adulto' o 'mayor de edad' cambia este valor por 18 de forma obligatoria. Si no se menciona edad mínima pon 'No mencionada') "
                #             "y 'edad_maxima' (debe ser un número entero o el string 'No mencionada'). "
                #             "Está prohibido escribir introducciones, explicaciones o notas. Devuelve solo el JSON crudo.")
                prompt_sistema = (
                    "Eres un analizador de texto estricto y un extractor de datos de alta precisión en español. "
                    "Tu única tarea es identificar rangos de edad y categorizar actividades.\n\n"
                    
                    "REGLAS INSTRUCCIONALES OBLIGATORIAS:\n"
                    "1. Devuelve ÚNICAMENTE un objeto JSON plano con tres claves exactas: 'categoria', 'edad_minima' y 'edad_maxima'.\n"
                    "2. La clave 'categoria' debe ser obligatoriamente 'actividad al aire libre' o 'resto'.\n"
                    "3. Las claves 'edad_minima' y 'edad_maxima' deben ser números enteros (int). Si no se mencionan, su valor predeterminado es el string 'No mencionada'.\n"
                    "4. LÓGICA DE RANGOS: Ante expresiones que indiquen un rango numérico de edad (estructuras del tipo 'de X a Y', 'entre X y Y', 'X-Y años'), estás obligado a evaluar de forma matemática ambos extremos: el número menor de la expresión SIEMPRE debe asignarse a 'edad_minima' y el número mayor SIEMPRE debe asignarse a 'edad_maxima'. No omitas ningún extremo bajo ninguna circunstancia, ignorando signos de exclamación o énfasis del texto.\n"
                    "5. LÓGICA DE ADULTOS: Si el texto contiene los términos 'adulto', 'adultos' o 'mayor de edad', la 'edad_minima' debe establecerse automáticamente como el número entero 18.\n"
                    "6. RESTRICCIÓN DE FORMATO: Está totalmente prohibido incluir introducciones, explicaciones, notas aclaratorias o bloques de código de marcado markdown (como ```json). Tu respuesta debe ser exclusivamente el JSON en texto crudo."
                )
                messages = [
                    {
                        "role": "system",
                        "content": (
                            prompt_sistema
                        )
                    },
                    {
                        "role": "user",
                        "content": f"Texto a analizar:\n{texto}"
                    }
                ]
                
                try:
                    # El LLM procesa todo el texto largo y genera la estructura JSON directamente
                    outputs = llm_local(messages, max_new_tokens=150, temperature=0.1)
                    
                    # CORRECCIÓN DE PARSEO: 
                    # El pipeline de text-generation devuelve una lista. Extraemos el primer elemento [0]
                    # y luego accedemos a la clave 'generated_text'. Si es una estructura de chat,
                    # el contenido final se extrae de forma segura indexando correctamente.
                    if isinstance(outputs, list) and len(outputs) > 0:
                        # Extraemos el contenido del último mensaje generado por el asistente
                        if 'generated_text' in outputs[0] and isinstance(outputs[0]['generated_text'], list):
                            respuesta_raw = outputs[0]['generated_text'][-1]['content'].strip()
                        else:
                            # Formato plano por si el pipeline devuelve texto directo en vez de estructura de chat
                            respuesta_raw = outputs[0]['generated_text'].strip()
                    else:
                        raise ValueError("Estructura de salida del LLM inesperada")
                    
                    # Limpieza básica por si el modelo incluye bloques de código markdown
                    respuesta_raw = respuesta_raw.replace("```json", "").replace("```", "").strip()
                    
                    # Parseamos la respuesta de la IA
                    datos_ia = json.loads(respuesta_raw)
                    
                    # Inyectamos los resultados directamente en tu objeto original
                    activity['categoria'] = datos_ia.get('categoria', 'resto')
                    activity['edad_minima'] = datos_ia.get('edad_minima', 'No mencionada')
                    activity['edad_maxima'] = datos_ia.get('edad_maxima', 'No mencionada')
                    
                except Exception as e:
                    print(f"❌ Error crítico en la generación o parseo del LLM: {e}")
                    activity['categoria'] = "Error"
                    activity['edad_minima'] = "No mencionada"
                    activity['edad_maxima'] = "No mencionada"

            elapsed = time.time() - start_time
            
            # Mantenemos tus prints detallados de salida
            print(f"📍 ID: {activity['id']}...")
            print(f"📍 Título: {activity['title']}...")
            print(f"   Descripción: {activity['description']}...")
            print(f"   Categoría: {activity['categoria']}")
            print(f"   👶 Edad mínima: {activity['edad_minima']}")
            print(f"   👴 Edad máxima: {activity['edad_maxima']}")
            print(f"   ⏱️ Tiempo: {elapsed:.2f}s\n")
            
            # break # Terminamos el bucle al encontrar el ID correspondiente


In [13]:
id = "50316249-homo-argentum.json"
id = "50294177-bruja-maruxa.json"
# id = "50281255-calle-42.json"
process_activity_id(id)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Procesando actividad con ID: 50294177-bruja-maruxa.json - La bruja Maruxa

📍 ID: https://datos.madrid.es/egob/catalogo/tipo/evento/50294177-bruja-maruxa.json...
📍 Título: La bruja Maruxa...
   Descripción: En este cuentacuentos la Bruja Maruxa nos trae sus cuentos mágicos y fascinantes… Unión de arte y relatos…. Ven a descubrir a la bruja más rockera. Espectáctulo familiar para niños de 3 a 80 años!!!! Organiza: Compañía Teatral Max Estrella....
   Categoría: actividad al aire libre
   👶 Edad mínima: 3
   👴 Edad máxima: 80
   ⏱️ Tiempo: 161.65s



In [14]:
import os
import json
import time
import torch
from transformers import pipeline

def procesar_actividades_en_batch(actividades):
    """
    Recibe la lista completa de actividades JSON y las clasifica de golpe (Batch).
    """
    print("Cargando LLM local en memoria para procesamiento por lotes...")
    
    # Inicializamos el pipeline de text-generation de manera global o local
    llm_local = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    tiempo_inicio = time.time()
    
    # 1. Construir la lista masiva de prompts (uno para cada descripción)
    listado_prompts = []
    indices_validos = [] # Guardamos el orden para reinyectar los datos correctamente

    prompt_sistema = (
                    "Eres un analizador de texto estricto y un extractor de datos de alta precisión en español. "
                    "Tu única tarea es identificar rangos de edad y categorizar actividades.\n\n"
                    
                    "REGLAS INSTRUCCIONALES OBLIGATORIAS:\n"
                    "1. Devuelve ÚNICAMENTE un objeto JSON plano con tres claves exactas: 'categoria', 'edad_minima' y 'edad_maxima'.\n"
                    "2. La clave 'categoria' debe ser obligatoriamente 'actividad al aire libre' o 'resto'.\n"
                    "3. Las claves 'edad_minima' y 'edad_maxima' deben ser números enteros (int). Si no se mencionan, su valor predeterminado es el string 'No mencionada'.\n"
                    "4. LÓGICA DE RANGOS: Ante expresiones que indiquen un rango numérico de edad (estructuras del tipo 'de X a Y', 'entre X y Y', 'X-Y años'), estás obligado a evaluar de forma matemática ambos extremos: el número menor de la expresión SIEMPRE debe asignarse a 'edad_minima' y el número mayor SIEMPRE debe asignarse a 'edad_maxima'. No omitas ningún extremo bajo ninguna circunstancia, ignorando signos de exclamación o énfasis del texto.\n"
                    "5. LÓGICA DE ADULTOS: Si el texto contiene los términos 'adulto', 'adultos' o 'mayor de edad', la 'edad_minima' debe establecerse automáticamente como el número entero 18.\n"
                    "6. RESTRICCIÓN DE FORMATO: Está totalmente prohibido incluir introducciones, explicaciones, notas aclaratorias o bloques de código de marcado markdown (como ```json). Tu respuesta debe ser exclusivamente el JSON en texto crudo."
                )

    for idx, activity in enumerate(actividades):
        texto = activity.get('description', '').strip()
        if not texto:
            activity['categoria'] = "N/A"
            activity['edad_minima'] = "No mencionada"
            activity['edad_maxima'] = "No mencionada"
            continue
        
        # Estructura de chat requerida por Qwen
        messages = [
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": f"Texto a analizar:\n{texto}"}
        ]
        listado_prompts.append(messages)
        indices_validos.append(idx)

    if not listado_prompts:
        print("No hay textos válidos para procesar.")
        return actividades

    print(f"🚀 Enviando {len(listado_prompts)} textos al LLM en modo Batch...")

    # 2. Ejecutar la inferencia masiva en una sola línea de código
    # El parámetro batch_size le dice al modelo cuántos textos procesar en paralelo a la vez
    outputs = llm_local(listado_prompts, max_new_tokens=150, temperature=0.1, batch_size=16)

    print("Procesando respuestas generadas por la IA...")

    # 3. Mapear y parsear los resultados de vuelta a tu lista original
    for idx_prompt, out in enumerate(outputs):
        idx_original = indices_validos[idx_prompt]
        
        try:
            # Extraemos de forma segura el contenido textual de la respuesta generada
            if isinstance(out, list) and len(out) > 0:
                respuesta_raw = out[0]['generated_text'][-1]['content'].strip()
            else:
                respuesta_raw = out['generated_text'][-1]['content'].strip()
            
            # Limpieza de marcado markdown por seguridad
            respuesta_raw = respuesta_raw.replace("```json", "").replace("```", "").strip()
            
            # Convertimos el string en diccionario JSON
            datos_ia = json.loads(respuesta_raw)
            
            # Guardamos los resultados numéricos en tu objeto original
            actividades[idx_original]['categoria'] = datos_ia.get('categoria', 'resto')
            actividades[idx_original]['edad_minima'] = datos_ia.get('edad_minima', 'No mencionada')
            actividades[idx_original]['edad_maxima'] = datos_ia.get('edad_maxima', 'No mencionada')
            
        except Exception as e:
            print(f"❌ Error parseando lote en índice original {idx_original}: {e}")
            actividades[idx_original]['categoria'] = "Error"
            actividades[idx_original]['edad_minima'] = "No mencionada"
            actividades[idx_original]['edad_maxima'] = "No mencionada"

    print(f"⏱️ Tiempo total de procesamiento Batch: {time.time() - tiempo_inicio:.2f}s")
    return actividades


In [ ]:
ac_list = ["50316249-homo-argentum.json", "50294177-bruja-maruxa.json", "50281255-calle-42.json"]
seleccion_actividades = []  # Procesamos solo las primeras 100 para la prueba
for activity in actividades:
    if activity['app_id'] in ac_list:
        seleccion_actividades.append(activity)

procesar_actividades_en_batch(seleccion_actividades)

Cargando LLM local en memoria para procesamiento por lotes...


Loading weights: 100%|██████████| 338/338 [00:16<00:00, 21.01it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


🚀 Enviando 3 textos al LLM en modo Batch...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Paso 7: Procesar actividad completa (todos los campos)

In [ ]:
print("🔬 Procesando actividad completa con todos los campos...\n")

# Seleccionar una actividad para análisis detallado
actividad = actividades_prueba[0]

print(f"📋 Actividad: {actividad.get('title')}")
print(f"📝 Descripción completa:")
print(f"{actividad.get('description', 'Sin descripción')}")
print("\n" + "="*60 + "\n")

# Preparar texto
title = actividad.get('title', '')
description = actividad.get('description', '')
clean_desc = clean_text(description)

print(f"🧹 Texto limpio ({len(clean_desc)} chars):")
print(f"{clean_desc[:200]}...\n")

# Procesar
print("🤖 Ejecutando modelos...\n")

# Clasificar aire libre
es_aire_libre = air_classifier.classify_single(clean_desc, title)
print(f"🌳 Es aire libre: {es_aire_libre}")

# Extraer edades
edades = age_extractor.extract_ages(clean_desc, title)
print(f"👶 Edad mínima: {edades['edad_minima']}")
print(f"👴 Edad máxima: {edades['edad_maxima']}")

# Resultado final
print("\n" + "="*60)
print("📊 RESULTADO FINAL:")
print("="*60)
resultado = {
    **actividad,
    'es_aire_libre': es_aire_libre,
    'edad_minima': edades['edad_minima'],
    'edad_maxima': edades['edad_maxima']
}
print(json.dumps({
    'title': resultado['title'],
    'es_aire_libre': resultado['es_aire_libre'],
    'edad_minima': resultado['edad_minima'],
    'edad_maxima': resultado['edad_maxima']
}, indent=2, ensure_ascii=False))

## Paso 8: Probar con texto personalizado

Prueba con tus propios textos para ver cómo funciona la extracción

In [ ]:
# Define tu propio texto de prueba
TEXTO_PRUEBA = """
Taller de pintura al aire libre en el parque del Retiro. 
Para niños y niñas de 6 a 12 años. 
Traer ropa cómoda y ganas de divertirse.
"""

TITULO_PRUEBA = "Taller de Pintura en el Retiro"

print(f"📝 Texto de prueba:")
print(f"{TEXTO_PRUEBA}\n")

# Limpiar
clean_test = clean_text(TEXTO_PRUEBA)

# Clasificar
es_aire_libre = air_classifier.classify_single(clean_test, TITULO_PRUEBA)
print(f"🌳 Es aire libre: {es_aire_libre}")

# Extraer edades
edades = age_extractor.extract_ages(clean_test, TITULO_PRUEBA)
print(f"👶 Edad mínima: {edades['edad_minima']}")
print(f"👴 Edad máxima: {edades['edad_maxima']}")

## Paso 9: Ejecutar script completo (opcional)

Descomenta para ejecutar el script completo con todas las actividades

In [ ]:
# Ejecutar script completo con límite de 10 actividades
# !python ../scripts/enrich_activities.py --limit 10 --output ../data/test_output.json

# Ver resultado
# if os.path.exists('../data/test_output.json'):
#     with open('../data/test_output.json', 'r') as f:
#         resultado = json.load(f)
#     print(f"✅ Procesadas {len(resultado)} actividades")
#     print("\nPrimer resultado:")
#     print(json.dumps(resultado[0], indent=2, ensure_ascii=False))

## 📊 Resumen y Estadísticas

In [ ]:
print("📈 ESTADÍSTICAS DEL MODELO")
print("="*60)

print("🤖 Modelos utilizados:")
print(f"   - Clasificación: {air_classifier.MODEL_NAME}")
print(f"   - QA Edades: {age_extractor.MODEL_NAME}")

print("\n⚙️ Configuración:")
print(f"   - Device: CPU")
print(f"   - Batch size: 32 (configurable)")

print("\n📋 Campos generados:")
print("   - es_aire_libre: booleano")
print("   - edad_minima: entero o null")
print("   - edad_maxima: entero o null")

print("\n💡 Notas:")
print("   - La primera ejecución descarga los modelos (~500MB)")
print("   - Tiempo estimado: 1-2 segundos por actividad")
print("   - Usa regex + IA para mejor precisión")